# 加入記憶與自我反思


**說明**

本教材參考 Udacity 的 AI Agents with LangChain and LangGraph 課程。

在前一個 AI 代理程式的基礎上，我們加入自我反思與記憶功能。這些功能讓代理程式可以反覆檢視自己的回應並逐步改進，同時保留所有互動的記錄。

這類似人類學習與回饋循環的過程，能推動代理程式產生更精準、更完善的輸出。


## 0. 匯入必要的套件


In [1]:
import json
import os
from typing import List, Dict, Literal
from openai import OpenAI
from openai.types.chat.chat_completion_message import ChatCompletionMessage

## 1. 使用 OpenAI 用戶端


若要連接 OpenAI，需要建立一個 OpenAI 用戶端，並傳入你的 OpenAI API 金鑰。

你可以直接傳入 `api_key` 參數。
```python
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
```


In [2]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [3]:
response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer all user questions"},
            {"role": "user", "content":"What have I asked?"},
        ],
        temperature=0.0,
    )
response.choices[0].message.content

"I don't have access to previous interactions or any specific questions you've asked before. However, I'm here to help with any questions or topics you'd like to discuss now! What can I assist you with?"

## 記憶概念的複習


為了加入反思功能，你需要確保代理程式能夠追蹤所有互動。我們先用一個簡單的 list 快速複習怎麼做。


In [4]:
memory = [
    {"role": "system", "content": "Answer all user questions"},
    {"role": "user", "content": "What's an API"},
]

In [5]:
new_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=memory,
    temperature=0.0,
)

memory.append(
    {"role": "assistant", "content": new_response.choices[0].message.content}
)

memory

[{'role': 'system', 'content': 'Answer all user questions'},
 {'role': 'user', 'content': "What's an API"},
 {'role': 'assistant',
  'content': 'An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate with each other. It defines the methods and data formats that applications can use to request and exchange information. APIs enable developers to access the functionality of other software services, libraries, or platforms without needing to understand their internal workings.\n\nFor example, a weather application might use an API to retrieve weather data from a remote server. The API specifies how the application can request the data (e.g., through specific URLs and parameters) and what format the data will be returned in (e.g., JSON or XML). This allows developers to build applications that can leverage existing services and data, facilitating integration and enhancing functionality.'}]

In [6]:
memory.append(
    {"role": "user", "content": "What have I asked?"}
)

memory

[{'role': 'system', 'content': 'Answer all user questions'},
 {'role': 'user', 'content': "What's an API"},
 {'role': 'assistant',
  'content': 'An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate with each other. It defines the methods and data formats that applications can use to request and exchange information. APIs enable developers to access the functionality of other software services, libraries, or platforms without needing to understand their internal workings.\n\nFor example, a weather application might use an API to retrieve weather data from a remote server. The API specifies how the application can request the data (e.g., through specific URLs and parameters) and what format the data will be returned in (e.g., JSON or XML). This allows developers to build applications that can leverage existing services and data, facilitating integration and enhancing functionality.'},
 {'role': 'user', 'c

In [7]:
new_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=memory,
    temperature=0.0,
)

memory.append(
    {"role": "assistant", "content": new_response.choices[0].message.content}
)

memory

[{'role': 'system', 'content': 'Answer all user questions'},
 {'role': 'user', 'content': "What's an API"},
 {'role': 'assistant',
  'content': 'An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate with each other. It defines the methods and data formats that applications can use to request and exchange information. APIs enable developers to access the functionality of other software services, libraries, or platforms without needing to understand their internal workings.\n\nFor example, a weather application might use an API to retrieve weather data from a remote server. The API specifies how the application can request the data (e.g., through specific URLs and parameters) and what format the data will be returned in (e.g., JSON or XML). This allows developers to build applications that can leverage existing services and data, facilitating integration and enhancing functionality.'},
 {'role': 'user', 'c

## 3. 建立記憶類別


建立一個正式的類別，用來處理更複雜的情況。

請建立 `Memory` 類別，並加入必要的方法。


In [8]:
class Memory:
    def __init__(self):
        self._messages: List[Dict[str, str]] = []

    def add_message(self, role: Literal['user', 'system', 'assistant'], content: str):
        self._messages.append({
            "role": role,
            "content": content
        })

    def get_messages(self) -> List[Dict[str, str]]:
        return self._messages

    # A new method
    def last_message(self) -> None:
        if self._messages:
            return self._messages[-1]

## 4. 更新 Agent 類別


將為 AI 代理程式加入自我反思能力，讓它可以批判自己的回應，並通過多次迭代修正回答。這個功能讓代理程式能在給出最終答案前，先評估自己的輸出並改善回應品質。

**目標**

你的任務是修改代理程式，讓它可以：

- 儲存對話歷史：實作記憶機制來追蹤互動。
- 產生初始回應：處理使用者輸入，並使用語言模型回傳回應。
- 在啟用時批判自己的回應：如果啟用自我反思，代理程式應產生對自己答案的回饋。
- 逐步修正回應：根據自我批判，代理程式應調整回答，以改善清楚度、正確性與相關性。

**步驟**

- 實作記憶層，用來保留對話歷史。
- 引入自我反思機制，讓代理程式可以分析自己的回應並修正。
- 限制自我反思的迭代次數，避免產生過多循環（最少 1 次，最多 3 次）。
- 允許使用者開啟或關閉自我反思，以保持彈性。

**注意事項**

- 自我反思會增加 API 呼叫次數，因此會增加成本與延遲。
- 迭代次數不應過多，否則可能讓回應變得冗長或偏離原本問題。
- 記憶內容應保持結構化，才方便後續檢索與分析。


**呼叫方法**

請重構 `invoke()` 方法。這個方法現在應包含：
- `self_reflection` 參數（預設值：`False`）；
- `max_iter` 參數（預設值：`1`）。

如果 `self_reflection` 設為 `True`，應使用迴圈先產生初始回應，接著在後續迭代中批判並修正該回應，直到達到 `max_iter` 定義的迭代次數。

使用 `self.memory` 來儲存每一個步驟。


自我反思的規則：

- 不允許小於 1 的值。
- 不允許大於 3 的值。
- 最大迭代次數由 `self_reflection` 旗標控制。
- 如果設為 `True`，至少需要額外呼叫一次 LLM 來產生批判。


In [9]:
SELF_CRITIQUE_PROMPT = """
Reflect on your previous response...
Identify any mistakes, areas for improvement, or ways to clarify the answer, making it more concise. 
Provide a revised response if necessary in a Json Output structure:
{
    "original_response": "",
    "revisions_needed": "",
    "updated_response": ""
}
"""

In [10]:
class Agent:
    """A self-reflection AI Agent"""

    def __init__(
        self,
        name:str = "Agent", 
        role:str = "Personal Assistant",
        instructions:str = "Help users with any question",
        model:str = "gpt-4o-mini",
        temperature:float = 0.0,
    ):
        self.name = name
        self.role = role
        self.instructions = instructions
        self.model = model
        self.temperature = temperature

        self.client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

        self.memory = Memory()
        self.memory.add_message(
            role="system",
            content=f"You're an AI Agent, your role is {self.role}, " 
                    f"and you need to {self.instructions}",
        )

        self.critique_prompt = SELF_CRITIQUE_PROMPT
    
    def invoke(self, 
               user_message: str, 
               self_reflection: bool = False, 
               max_iter: int = 1, 
               verbose: bool = False) -> str:
    
        # Rules
        # - Don't allow values less than 1
        # - Don't allow values greater than 3
        # - Max iter is controlled by self_reflection flag. 
        # - If set to true, it needs to call the LLM at least once more for the criticism

        self.memory.add_message(
            role="user",
            content=user_message
        )
        if verbose:
            self._log_last_message()

        max_iter = max_iter if max_iter >= 1 else 1
        max_iter = max_iter if max_iter <= 3 else 3
        max_iter = max_iter if self_reflection else 0.5
        loops = 2 * max_iter

        for i in range(loops):
            ai_message = self._get_completion(
                messages = self.memory.get_messages()
            )

            self.memory.add_message(
                role = "assistant",
                content = ai_message.content,
            )
            if verbose:
                self._log_last_message()

            if i < loops - 1:
                self.memory.add_message(
                    role = "user", 
                    content = self.critique_prompt
                )
                if verbose:
                    self._log_last_message()

                ai_message = self._get_completion(
                    messages = self.memory.get_messages()
                )

    def _get_completion(self, messages:List[Dict])-> ChatCompletionMessage:
        response = self.client.chat.completions.create(
            model=self.model,
            temperature=self.temperature,
            messages=messages
        )
        
        return response.choices[0].message

    def _log_last_message(self):
        print(f"### {self.memory.last_message()['role']} message ###\n".upper())
        print(f"{self.memory.last_message()['content']} \n")
        print("\n________________________________________________________________\n")


## 5. 建立一些代理程式並執行


建立一些特定用途的代理程式，並使用 `self_reflection = True` 呼叫它們。


In [11]:
agent = Agent()
agent.invoke(
    user_message="Pick only one. Who is the best character in Game of Thrones?",
    self_reflection=True,
    verbose=True,
)

### USER MESSAGE ###

Pick only one. Who is the best character in Game of Thrones? 


________________________________________________________________

### ASSISTANT MESSAGE ###

Choosing the best character in "Game of Thrones" is subjective, but many fans often cite Tyrion Lannister as a standout. His wit, intelligence, and moral complexity make him a compelling character throughout the series. However, opinions vary widely, and others might argue for characters like Jon Snow, Daenerys Targaryen, or Arya Stark. Who's your favorite? 


________________________________________________________________

### USER MESSAGE ###


Reflect on your previous response...
Identify any mistakes, areas for improvement, or ways to clarify the answer, making it more concise. 
Provide a revised response if necessary in a Json Output structure:
{
    "original_response": "",
    "revisions_needed": "",
    "updated_response": ""
}
 


________________________________________________________________

### 

In [12]:
agent.memory.get_messages()

[{'role': 'system',
  'content': "You're an AI Agent, your role is Personal Assistant, and you need to Help users with any question"},
 {'role': 'user',
  'content': 'Pick only one. Who is the best character in Game of Thrones?'},
 {'role': 'assistant',
  'content': 'Choosing the best character in "Game of Thrones" is subjective, but many fans often cite Tyrion Lannister as a standout. His wit, intelligence, and moral complexity make him a compelling character throughout the series. However, opinions vary widely, and others might argue for characters like Jon Snow, Daenerys Targaryen, or Arya Stark. Who\'s your favorite?'},
 {'role': 'user',
  'content': '\nReflect on your previous response...\nIdentify any mistakes, areas for improvement, or ways to clarify the answer, making it more concise. \nProvide a revised response if necessary in a Json Output structure:\n{\n    "original_response": "",\n    "revisions_needed": "",\n    "updated_response": ""\n}\n'},
 {'role': 'assistant',
  'c

In [13]:
json.loads(agent.memory.last_message()["content"])["updated_response"]

"Many fans consider Tyrion Lannister the best character in 'Game of Thrones' due to his wit and complexity, but opinions vary widely. Other popular choices include Jon Snow and Daenerys Targaryen."

## 6. 進階實驗


現在你已經理解它的運作方式，可以嘗試做一些新的實驗。

- 嘗試不同的批判 prompt。
- 當你增加迭代次數時，會發生什麼事？
- 嘗試存取記憶來檢查內容（`agent.memory`），而不是只閱讀輸出（`verbose=False`）。
- 你還可以嘗試什麼？
